# Words and the index

[Notebook 1](01-building-a-grid.ipynb) built a puzzle and treated the
dictionary as a given. This is where the words come from, what is done to them,
and how they are stored so that a search can use them millions of times without
slowing down.

1. The raw list, and what normalising it costs
2. Which words are allowed in a puzzle at all
3. How familiar a word is, and why that is measured as a rank
4. The three dials a setter has, and what each one does
5. The index: matching a pattern with integer arithmetic

In [1]:
import os
import re
import sys
import time
from collections import Counter

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from crossword.index import Index
from crossword.words import load

## 1. The raw list

UKACD, the UK Advanced Cryptics Dictionary: about a quarter of a million
entries assembled for exactly this purpose, so it carries the phrases and
proper nouns a cryptic actually uses rather than only the headwords of a
dictionary.

Every entry is normalised into two forms. `text` is the fill string, lowercase
letters only, which is what goes in the grid. `surface` keeps one original
spelling.

Keeping both is not tidiness. Folding `twelfth night` down to `twelfthnight`
destroys the information a solver needs — the enumeration (7,5) — and it cannot
be recovered from the fill string afterwards.

In [2]:
entries = load("crossword/UKACD.txt")
print(f"{len(entries):,} entries after the default filtering\n")

interesting = [e for e in entries if e.phrase][:3] + \
              [e for e in entries if "-" in e.surface][:2]
for entry in entries[:2] + interesting:
    print(f"  text={entry.text!r:18} surface={entry.surface!r:22} "
          f"proper={entry.proper} phrase={entry.phrase}")

221,835 entries after the default filtering

  text='aardvark'         surface='aardvark'             proper=False phrase=False
  text='aardvarks'        surface='aardvarks'            proper=False phrase=False
  text='abadegg'          surface='a bad egg'            proper=False phrase=True
  text='abadhat'          surface='a bad hat'            proper=False phrase=True
  text='abandonship'      surface='abandon ship'         proper=False phrase=True
  text='abatjour'         surface='abat-jour'            proper=False phrase=True
  text='abatjours'        surface='abat-jours'           proper=False phrase=True


`phrase` is true whenever the text differs from the surface, which catches
spaces, hyphens and apostrophes in one flag. That matters more than it looks:
stripping punctuation turns `it'll` into `itll` and `ro-ros` into `roros`, and
those are perfectly good crossword answers but nobody's idea of a word.

In [3]:
strange = [e for e in entries if e.phrase and "'" in e.surface][:5]
for entry in strange:
    print(f"  {entry.text:12} <- {entry.surface}")

print(f"\nwith phrases excluded: "
      f"{len(load('crossword/UKACD.txt', allow_phrases=False)):,} entries")

  abody        <- a'body
  aboveoneshead <- above one's head
  adderstongue <- adder's-tongue
  affairedamour <- affaire d'amour
  affairedhonneur <- affaire d'honneur

with phrases excluded: 179,710 entries


## 2. Which words are allowed

`load` takes the filters that decide the vocabulary before anything else
happens. Proper nouns are excluded by default; phrases are not, because a
cryptic uses them freely.

In [4]:
for label, kwargs in (
    ("default", {}),
    ("no phrases", {"allow_phrases": False}),
    ("with proper nouns", {"allow_proper": True}),
    ("4 to 12 letters only", {"min_length": 4, "max_length": 12}),
):
    print(f"  {label:22} {len(load('crossword/UKACD.txt', **kwargs)):>8,}")

  default                 221,835
  no phrases              179,710
  with proper nouns       241,579
  4 to 12 letters only    196,830


## 3. How familiar a word is

A dictionary of 221,835 words will happily fill a grid with entries no solver
has met. To prefer the ordinary ones, each word gets a score, built once and
stored in `crossword/scores.txt`:

    score = max( log1p(Guardian answer uses), 0.75 x wordfreq Zipf )

Two sources, because each fails alone. General frequency does not know that
`isle` and `oleo` are crossword staples out of all proportion to their use in
English. The Guardian answer counts do not know that `windscreen` and
`pavement` are ordinary words that simply have not come up. Taking the larger
of the two lets either source rescue a word.

In [5]:
scores = {}
with open("crossword/scores.txt", encoding="utf-8") as handle:
    for line in handle:
        if not line.startswith("#") and line.strip():
            word, value = line.split()
            scores[word] = float(value)

print(f"{len(scores):,} words carry a score\n")
for word in ("isle", "oleo", "pavement", "windscreen", "abaci", "zygote"):
    print(f"  {word:12} {scores.get(word, 0.0):5.2f}")

105,373 words carry a score

  isle          4.37
  oleo          1.48
  pavement      2.71
  windscreen    2.18
  abaci         2.20
  zygote        1.76


### Why the score becomes a rank

Raw scores cannot be compared across lengths. Long words are rarer than short
ones by nature, so a fifteen-letter word with a middling score may be the most
ordinary fifteen-letter word there is.

So the index converts each score into a **quantile**: the word's rank among
words *of its own length*, from 0 for the most obscure to 1 for the most
ordinary. Published answers turn out to sit at a remarkably steady quantile
whatever their length, which is what makes a single target workable for a whole
grid.

In [6]:
index = Index(entries, scores)

for length in (4, 8, 12):
    bucket = index.lengths[length]
    ranked = sorted(range(len(bucket.words)), key=lambda i: bucket.quantile[i])
    bottom = bucket.words[ranked[len(ranked) // 20]].upper()
    middle = bucket.words[ranked[len(ranked) // 2]].upper()
    top = bucket.words[ranked[-1]].upper()
    print(f"  {length:2d} letters, {len(bucket.words):>6,} words:  "
          f"5% {bottom:14} 50% {middle:14} top {top}")

   4 letters,  4,582 words:  5% SHMO           50% BEDE           top THAT
   8 letters, 32,192 words:  5% BUSKINGS       50% COTELINE       top BUSINESS
  12 letters, 17,661 words:  5% BOTTLEBLONDS   50% SAVEONESSKIN   top RELATIONSHIP


## 4. The three dials

These get confused, so they are worth separating. They act at different times
and do different things.

| dial | when | what it does |
|---|---|---|
| `--min-score` | before the search | **removes** words from the vocabulary entirely |
| `--max-uses` | before the search | **removes** words used too often as answers |
| `--aim` | during the search | the familiarity rank to aim *at* |
| `--commonness` | during the search | how hard to pull towards that aim |

`--min-score` and `--max-uses` are hard cuts. They change what exists. A floor
costs coverage quickly, and it is easy to cut so deep that nothing fills.

In [7]:
from crossword import frequency

counts = frequency.load("crossword/frequency.txt")
print("floor       words     15-letter words")
for floor in (0.0, 1.0, 2.0, 3.0):
    kept = frequency.select(entries, scores, counts, min_score=floor)
    long = sum(1 for e in kept if len(e.text) == 15)
    print(f"  {floor:4.1f}   {len(kept):>8,}   {long:>8,}")
print("\nA 15x15 grid needs several long entries, so a floor of 2.0 is")
print("already close to unfillable and 3.0 is past it.")

floor       words     15-letter words
   0.0    221,835      4,426
   1.0     84,268        895
   2.0     33,687        125
   3.0      6,757          4

A 15x15 grid needs several long entries, so a floor of 2.0 is
already close to unfillable and 3.0 is past it.


`--aim` and `--commonness` do not remove anything. Every word stays reachable;
the search simply prefers some over others. That distinction is what keeps a
grid fillable: when the crossings leave exactly one obscure word that works, a
preference lets it through and a filter does not.

**`aim` says where, `commonness` says how hard.** Holding the aim at 0.85 and
varying only the pull, on one grid over twelve seeds:

    commonness   mean rank   spread
        0.0         0.51      0.28     familiarity ignored entirely
        1.0         0.69      0.23
        3.0         0.79      0.16     the default
        8.0         0.81      0.14
       20.0         0.81      0.12     saturated

Raising it tightens the fill around the target rather than simply raising it,
and the effect saturates: past about 3 the mean stops moving, because the
remaining variation is not preference but necessity -- words the crossings
leave no choice about.

On a hard grid it matters less than one would hope. The same sweep on a sparse
American grid, every letter checked, gave a mean rank of 0.53 at commonness 0
and 0.53 to 0.62 at every setting above it, with completions of 3, 5, 3 and 6
out of 10 -- differences well inside the noise of ten seeds. The crossings, not
the preference, decide most of what goes in a tightly checked grid.

So the answer to "do I need all three" is that they answer different questions.
Use `--min-score` when a word must not appear at all. Use `--aim` to say what
register the puzzle is in. Leave `--commonness` alone unless you want
familiarity switched off, which is what 0 does.

## 5. The index: matching a pattern with integers

The filler asks one question, millions of times: *given a slot with some
letters known, which words still fit?*

Number the five-letter words 0, 1, 2, ... Then a **set** of words is a single
integer, with bit *i* set when word *i* is in the set. For every position and
every letter, the index stores one such integer: the set of words having that
letter in that position.

A pattern is the intersection of one set per known letter, and intersecting
bitsets is bitwise AND:

    S _ E _ L   ->   (S in position 0)  &  (E in position 2)  &  (L in position 4)

One AND per known letter, regardless of dictionary size, and the result is
exactly the set of candidates. Counting them is a popcount. Python integers are
arbitrary-precision, so a 10,162-word set is a 10,162-bit number and the
arithmetic happens in C.

In [8]:
bucket = index.lengths[5]
print(f"{len(bucket.words):,} five-letter words")

mask = bucket.match("s.e.l")
print(f"\nS_E_L  ->  {bucket.count('s.e.l')} candidates")
print("  ", ", ".join(bucket.iterate(mask)))
print(f"\nthe mask is one integer of {mask.bit_length():,} bits, "
      f"{bin(mask).count('1')} set")

10,162 five-letter words

S_E_L  ->  16 candidates
   sheal, sheel, shell, sheol, skell, smell, snell, speal, speel, spell, steal, steel, steil, stell, sweal, swell

the mask is one integer of 8,556 bits, 16 set


### What it buys

In [9]:
words = bucket.words
began = time.time()
for _ in range(200):
    naive = [w for w in words if re.match("^s.e.l$", w)]
scan = (time.time() - began) / 200

began = time.time()
for _ in range(200):
    fast = list(bucket.iterate(bucket.match("s.e.l")))
bits = (time.time() - began) / 200

print(f"scanning every word:  {scan * 1e6:7.0f} us")
print(f"bitset intersection:  {bits * 1e6:7.0f} us")
print(f"same answer: {sorted(naive) == sorted(fast)}   "
      f"speedup: {scan / bits:.0f}x")

scanning every word:     3389 us
bitset intersection:       12 us
same answer: True   speedup: 285x


### One universe per length

Word ids are assigned within a length, so bit 17 means a different word in the
five-letter index than in the six-letter one. That costs nothing, because a
slot of five cells can only be filled by a five-letter word: the two sets never
need to be compared. It also keeps every integer as small as it can be, and the
cost of the AND grows with the size of the integer.

In [10]:
print("length   words    bits per mask")
for length in (3, 5, 8, 12, 15):
    n = len(index.lengths[length].words)
    print(f"  {length:3d}   {n:>7,}   {n:>10,}")

length   words    bits per mask
    3     1,112        1,112
    5    10,162       10,162
    8    32,192       32,192
   12    17,661       17,661
   15     4,426        4,426


## Next

- **[the fill search](03-the-fill-search.ipynb)**: how these candidate sets are
  used to choose words and to know when to give up
- back to **[building a grid](01-building-a-grid.ipynb)** for the whole
  process